<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/llama_3_2_3b_fine_tuning_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A100 GPU + High Memory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()

In [ ]:

# Install in correct order - accelerate first, then transformers
!pip install accelerate
!pip install transformers
!pip install datasets scikit-learn matplotlib bitsandbytes

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    TrainerCallback
)
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

# Verify versions
import transformers
print(f"Transformers version: {transformers.__version__}")

class TimeTrackerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        self.start_time = time.time()

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        elapsed_time = time.time() - self.start_time
        print(f"Epoch {state.epoch} training time: {elapsed_time:.2f} seconds")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 1,039

In [ ]:
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

df = pd.read_csv('/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-1039.csv', sep=',')
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['inflation'].value_counts()}")

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=post)

df['formatted_body'] = df['body'].apply(format_with_prompt)

train_df, val_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['inflation'])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

# Check if classification head is properly initialized
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [ ]:
model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

# Check if classification head is properly initialized
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

training_args = TrainingArguments(
    # ☆
    output_dir="/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-rerun",
    logging_dir="/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-rerun/logs",
    # ☆
    # evaluation_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    # ☆
    logging_steps=10,
    learning_rate=5e-5,
    # ☆
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    gradient_accumulation_steps=4,
    fp16=False,
    bf16=True,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    seed=42,
    run_name="llama-3.2-3b-inflation-1039",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # processing_class=tokenizer,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training...")
trainer.train()

# 520

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch
import gc
import os

# 1. Clear GPU Memory (Important for re-runs)
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory Cleared.")

# 2. Prompt Definition
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=post)

# 3. Load Dataset (N=520)
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-520.csv'
df = pd.read_csv(file_path, sep=',')
print(f"Dataset shape: {df.shape}")

# 4. Apply Formatting and Split
df['formatted_body'] = df['body'].apply(format_with_prompt)
train_df, val_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['inflation'])

# 5. Convert to HF Datasets
raw_train_dataset = Dataset.from_pandas(train_df)
raw_val_dataset = Dataset.from_pandas(val_df)
print("Data loaded and split successfully.")

In [ ]:
from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Tokenize with truncation and padding
    return tokenizer(
        examples["formatted_body"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("Tokenizing datasets...")
tokenized_train = raw_train_dataset.map(tokenize_function, batched=True)
tokenized_val = raw_val_dataset.map(tokenize_function, batched=True)

# Remove unused columns for training
columns_to_remove = ['body', 'formatted_body']
if '__index_level_0__' in tokenized_train.column_names:
    columns_to_remove.append('__index_level_0__')

tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_val = tokenized_val.remove_columns(columns_to_remove)

# Rename labels and set format
tokenized_train = tokenized_train.rename_column("inflation", "labels")
tokenized_val = tokenized_val.rename_column("inflation", "labels")
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print("Tokenization complete.")

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
print(f"Model loaded on: {model.device}")

In [ ]:
import numpy as np
import time
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import TrainerCallback

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

class TimeTrackerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        self.start_time = time.time()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        elapsed_time = time.time() - self.start_time
        print(f"Epoch {state.epoch} training time: {elapsed_time:.2f} seconds")

In [ ]:
# Temporary local directory (Faster & Safe)
local_output_dir = "/content/temp_llama_520"

training_args = TrainingArguments(
    output_dir=local_output_dir,
    logging_dir=f"{local_output_dir}/logs",

    # N=520 Settings
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=5e-5,
    num_train_epochs=8, # N=520 -> 8 Epochs

    # Standard A100 Settings
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="adamw_torch",

    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    fp16=False,
    bf16=True,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    seed=42,
    run_name="llama-3.2-3b-inflation-520",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training for N=520 (Saving locally first)...")
trainer.train()

# ---------------------------------------------------------
# Save to Drive (Crucial Step)
# ---------------------------------------------------------
print("Training finished. Copying BEST model to Google Drive...")

# Final destination on Drive
drive_save_path = "/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-520-rerun"

# Ensure directory exists
if not os.path.exists(drive_save_path):
    os.makedirs(drive_save_path)

# Save the best model loaded by load_best_model_at_end
trainer.save_model(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"Success! Model saved to: {drive_save_path}")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive synced and unmounted. Now safe to restart.")

# 260

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch
import gc
import os

# 1. Clear GPU Memory (Important for re-runs)
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory Cleared.")

# 2. Prompt Definition
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=post)

# 3. Load Dataset (N=520)
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-260.csv'
df = pd.read_csv(file_path, sep=',')
print(f"Dataset shape: {df.shape}")

# 4. Apply Formatting and Split
df['formatted_body'] = df['body'].apply(format_with_prompt)
train_df, val_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['inflation'])

# 5. Convert to HF Datasets
raw_train_dataset = Dataset.from_pandas(train_df)
raw_val_dataset = Dataset.from_pandas(val_df)
print("Data loaded and split successfully.")

from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Tokenize with truncation and padding
    return tokenizer(
        examples["formatted_body"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("Tokenizing datasets...")
tokenized_train = raw_train_dataset.map(tokenize_function, batched=True)
tokenized_val = raw_val_dataset.map(tokenize_function, batched=True)

# Remove unused columns for training
columns_to_remove = ['body', 'formatted_body']
if '__index_level_0__' in tokenized_train.column_names:
    columns_to_remove.append('__index_level_0__')

tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_val = tokenized_val.remove_columns(columns_to_remove)

# Rename labels and set format
tokenized_train = tokenized_train.rename_column("inflation", "labels")
tokenized_val = tokenized_val.rename_column("inflation", "labels")
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print("Tokenization complete.")

from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
print(f"Model loaded on: {model.device}")

import numpy as np
import time
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import TrainerCallback

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

class TimeTrackerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        self.start_time = time.time()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        elapsed_time = time.time() - self.start_time
        print(f"Epoch {state.epoch} training time: {elapsed_time:.2f} seconds")



In [ ]:
import os

# ---------------------------------------------------------
# Disk Space Safety Measure
# ---------------------------------------------------------
# 1. Temporary Local Directory (Fast & Safe)
local_output_dir = "/content/temp_llama_260"

# 2. Final Destination on Google Drive (Only for the best model)
final_drive_output_dir = "/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-260-rerun"

training_args = TrainingArguments(
    # 【Modified】Save to local disk first
    output_dir=local_output_dir,
    logging_dir=f"{local_output_dir}/logs",

    # Strategy settings
    eval_strategy="epoch",
    save_strategy="epoch",

    # Reduce logging steps since 1 epoch has fewer steps
    logging_steps=5,

    learning_rate=5e-5,

    # Scaling Law: Double the epochs (8 -> 16)
    num_train_epochs=16,

    # Standard settings for A100 (Keep consistent with N=1039)
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="adamw_torch",

    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    fp16=False,
    bf16=True, # A100 supports BF16
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    seed=42,

    # Update run name
    run_name="llama-3.2-3b-inflation-260",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training for N=260 (Saving locally to avoid Drive errors)...")
trainer.train()

# ---------------------------------------------------------
# Post-Training: Transfer Best Model to Drive
# ---------------------------------------------------------
print("Training finished. Saving the BEST model to Google Drive...")

# Create directory if it doesn't exist
if not os.path.exists(final_drive_output_dir):
    os.makedirs(final_drive_output_dir)

# Save the model loaded by load_best_model_at_end
trainer.save_model(final_drive_output_dir)
tokenizer.save_pretrained(final_drive_output_dir)

print(f" Success! Best model saved to: {final_drive_output_dir}")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive synced and unmounted. Now safe to restart.")

# 130

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch
import gc
import os

# 1. Clear GPU Memory (Important for re-runs)
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory Cleared.")

# 2. Prompt Definition
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=post)

# 3. Load Dataset (N=520)
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-130.csv'
df = pd.read_csv(file_path, sep=',')
print(f"Dataset shape: {df.shape}")

# 4. Apply Formatting and Split
df['formatted_body'] = df['body'].apply(format_with_prompt)
train_df, val_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['inflation'])

# 5. Convert to HF Datasets
raw_train_dataset = Dataset.from_pandas(train_df)
raw_val_dataset = Dataset.from_pandas(val_df)
print("Data loaded and split successfully.")

from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Tokenize with truncation and padding
    return tokenizer(
        examples["formatted_body"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("Tokenizing datasets...")
tokenized_train = raw_train_dataset.map(tokenize_function, batched=True)
tokenized_val = raw_val_dataset.map(tokenize_function, batched=True)

# Remove unused columns for training
columns_to_remove = ['body', 'formatted_body']
if '__index_level_0__' in tokenized_train.column_names:
    columns_to_remove.append('__index_level_0__')

tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_val = tokenized_val.remove_columns(columns_to_remove)

# Rename labels and set format
tokenized_train = tokenized_train.rename_column("inflation", "labels")
tokenized_val = tokenized_val.rename_column("inflation", "labels")
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print("Tokenization complete.")

from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
print(f"Model loaded on: {model.device}")

import numpy as np
import time
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import TrainerCallback

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

class TimeTrackerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        self.start_time = time.time()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        elapsed_time = time.time() - self.start_time
        print(f"Epoch {state.epoch} training time: {elapsed_time:.2f} seconds")



In [ ]:
import os

# ---------------------------------------------------------
# Disk Space Safety Measure for N=130
# ---------------------------------------------------------
# 1. Temporary Local Directory
local_output_dir = "/content/temp_llama_130"

# 2. Final Destination on Google Drive
final_drive_output_dir = "/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-130-rerun"

training_args = TrainingArguments(
    # 【Modified】Save to local disk first
    output_dir=local_output_dir,
    logging_dir=f"{local_output_dir}/logs",

    # Strategy settings: Switch to 'steps' for small data
    eval_strategy="steps",
    save_strategy="steps",

    # Evaluate and Save every 20 steps (approx every 3 epochs)
    eval_steps=20,
    save_steps=20,

    # Log frequently
    logging_steps=5,

    learning_rate=5e-5,

    # Scaling Law: Double the epochs (16 -> 32)
    num_train_epochs=32,

    # Standard settings for A100 (Keep consistent with N=1039)
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="adamw_torch",

    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    fp16=False,
    bf16=True, # A100 supports BF16
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    seed=42,

    # Update run name
    run_name="llama-3.2-3b-inflation-130",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training for N=130 (Saving locally)...")
trainer.train()

# ---------------------------------------------------------
# Post-Training: Transfer Best Model to Drive
# ---------------------------------------------------------
print("Training finished. Saving the BEST model to Google Drive...")

# Create directory if it doesn't exist
if not os.path.exists(final_drive_output_dir):
    os.makedirs(final_drive_output_dir)

# Save the model loaded by load_best_model_at_end
trainer.save_model(final_drive_output_dir)
tokenizer.save_pretrained(final_drive_output_dir)

print(f" Success! Best model saved to: {final_drive_output_dir}")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive synced and unmounted. Now safe to restart.")

# 65

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch
import gc
import os

# 1. Clear GPU Memory (Important for re-runs)
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory Cleared.")

# 2. Prompt Definition
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=post)

# 3. Load Dataset (N=520)
file_path = '/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-65.csv'
df = pd.read_csv(file_path, sep=',')
print(f"Dataset shape: {df.shape}")

# 4. Apply Formatting and Split
df['formatted_body'] = df['body'].apply(format_with_prompt)
train_df, val_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['inflation'])

# 5. Convert to HF Datasets
raw_train_dataset = Dataset.from_pandas(train_df)
raw_val_dataset = Dataset.from_pandas(val_df)
print("Data loaded and split successfully.")

from transformers import AutoTokenizer

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Tokenize with truncation and padding
    return tokenizer(
        examples["formatted_body"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("Tokenizing datasets...")
tokenized_train = raw_train_dataset.map(tokenize_function, batched=True)
tokenized_val = raw_val_dataset.map(tokenize_function, batched=True)

# Remove unused columns for training
columns_to_remove = ['body', 'formatted_body']
if '__index_level_0__' in tokenized_train.column_names:
    columns_to_remove.append('__index_level_0__')

tokenized_train = tokenized_train.remove_columns(columns_to_remove)
tokenized_val = tokenized_val.remove_columns(columns_to_remove)

# Rename labels and set format
tokenized_train = tokenized_train.rename_column("inflation", "labels")
tokenized_val = tokenized_val.rename_column("inflation", "labels")
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print("Tokenization complete.")

from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
print(f"Model loaded on: {model.device}")

import numpy as np
import time
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import TrainerCallback

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

class TimeTrackerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        self.start_time = time.time()
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        elapsed_time = time.time() - self.start_time
        print(f"Epoch {state.epoch} training time: {elapsed_time:.2f} seconds")



In [ ]:
import os

# ---------------------------------------------------------
# Disk Space Safety Measure for N=65
# ---------------------------------------------------------
# 1. Temporary Local Directory
local_output_dir = "/content/temp_llama_65"

# 2. Final Destination on Google Drive
final_drive_output_dir = "/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-65-rerun/final_best"

training_args = TrainingArguments(
    # 【Modified】Save to local disk first
    output_dir=local_output_dir,
    logging_dir=f"{local_output_dir}/logs",

    # Strategy settings: Use 'steps' to avoid excessive saving
    eval_strategy="steps",
    save_strategy="steps",

    # Evaluate and Save every 20 steps (approx every 6-7 epochs)
    eval_steps=20,
    save_steps=20,

    # Log frequently
    logging_steps=5,

    learning_rate=5e-5,

    # Scaling Law: Double the epochs (32 -> 64)
    num_train_epochs=64,

    # Standard settings for A100 (Keep consistent with N=1039)
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="adamw_torch",

    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    fp16=False,
    bf16=True, # A100 supports BF16
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    seed=42,

    # Update run name
    run_name="llama-3.2-3b-inflation-65",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training for N=65 (Saving locally)...")
trainer.train()

# ---------------------------------------------------------
# Post-Training: Transfer Best Model to Drive
# ---------------------------------------------------------
print("Training finished. Saving the BEST model to Google Drive...")

# Create directory if it doesn't exist
if not os.path.exists(final_drive_output_dir):
    os.makedirs(final_drive_output_dir)

# Save the model loaded by load_best_model_at_end
trainer.save_model(final_drive_output_dir)
tokenizer.save_pretrained(final_drive_output_dir)

print(f" Success! Best model saved to: {final_drive_output_dir}")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Drive synced and unmounted. Now safe to restart.")

# For Check (1,039)

In [ ]:
INFLATION_PROMPT = """You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories: 0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market. 2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble. 1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text. Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.

Reddit Post: {post}

Classification:"""

df = pd.read_csv('/content/drive/MyDrive/world-inflation/data/reddit/production/2-main-prod-1039.csv', sep=',')
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['inflation'].value_counts()}")

def format_with_prompt(post):
    return INFLATION_PROMPT.format(post=post)

df['formatted_body'] = df['body'].apply(format_with_prompt)

train_df, val_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df['inflation'])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

# Check if classification head is properly initialized
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [ ]:
model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# === Build tokenized_train / tokenized_val for Trainer ===

# 1) Drop the pandas index column that Dataset.from_pandas often adds
for ds_name, ds in [("train_dataset", train_dataset), ("val_dataset", val_dataset)]:
    if "__index_level_0__" in ds.column_names:
        if ds_name == "train_dataset":
            train_dataset = ds.remove_columns(["__index_level_0__"])
        else:
            val_dataset = ds.remove_columns(["__index_level_0__"])

# 2) Ensure label column name is "labels" (Trainer expects this by default)
if "inflation" in train_dataset.column_names:
    train_dataset = train_dataset.rename_column("inflation", "labels")
if "inflation" in val_dataset.column_names:
    val_dataset = val_dataset.rename_column("inflation", "labels")

# 3) Keep only the text + labels columns (avoid accidental leakage / unused columns issues)
keep_cols = ["formatted_body", "labels"]
train_dataset = train_dataset.remove_columns([c for c in train_dataset.column_names if c not in keep_cols])
val_dataset   = val_dataset.remove_columns([c for c in val_dataset.column_names   if c not in keep_cols])

# 4) Tokenize
MAX_LEN = 512  # adjust if you need longer prompts; keep moderate to avoid OOM
def tokenize_fn(batch):
    return tokenizer(
        batch["formatted_body"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["formatted_body"])
tokenized_val   = val_dataset.map(tokenize_fn, batched=True, remove_columns=["formatted_body"])

# 5) Make sure labels are int (SequenceClassification expects integer class ids)
tokenized_train = tokenized_train.cast_column("labels", tokenized_train.features["labels"])
tokenized_val   = tokenized_val.cast_column("labels", tokenized_val.features["labels"])

# 6) Set torch format for Trainer
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("tokenized_train columns:", tokenized_train.column_names)
print("tokenized_val columns:", tokenized_val.column_names)
print("Example labels:", tokenized_train[0]["labels"])

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    token=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    problem_type="single_label_classification"
)

model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

# Check if classification head is properly initialized
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

training_args = TrainingArguments(
    # ☆
    output_dir="/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-rerun-check",
    logging_dir="/content/drive/MyDrive/world-inflation/data/model/llama-3.2-3b-fine-tuning-rerun-check/logs",
    # ☆
    # evaluation_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    # ☆
    logging_steps=10,
    learning_rate=5e-5,
    # ☆
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    gradient_accumulation_steps=4,
    fp16=False,
    bf16=True,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    seed=42,
    run_name="llama-3.2-3b-inflation-1039",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # processing_class=tokenizer,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TimeTrackerCallback()]
)

print("Starting training...")
trainer.train()